# AIoT-DA Lecture 6: Multiple Linear Regression on 50 Startups Dataset
### Applying the CRISP-DM Methodology
**Author:** Hank Chang (張佑維)  
**Topic 1:** 50 Startups Profit Prediction & Feature Selection Optimization  
**Interactive Web Explorer:** [https://hankyyyyyy.github.io/Hank_Chang_PersonalPage/AIOT/](https://hankyyyyyy.github.io/Hank_Chang_PersonalPage/AIOT/)

---

## 📌 CRISP-DM 6 大階段目錄
1. **Phase 1: Business Understanding (商業理解)**
2. **Phase 2: Data Understanding (資料理解與探索)**
3. **Phase 3: Data Preparation (資料前處理與編碼)**
4. **Phase 4: Modeling (模型建置與特徵選取)**
5. **Phase 5: Evaluation (模型評估與奧坎剃刀驗證)**
6. **Phase 6: Deployment (模型序列化導出與部署)**

## [Phase 1] Business Understanding (商業理解)
- **商業目標 (Objective):** 預測創業公司之淨利 (Profit)，找出影響收益之關鍵推動因子 (R&D Spend, Marketing, Administration, State)。
- **利害關係人 (Stakeholders):** 創投機構 (VC)、天使投資人、企業決策層。
- **成功指標 (Success Criteria):** 達到極高判定係數 (R² > 0.94) 並最小化測試集均方根誤差 (RMSE < $8,500)，辨識並剔除過擬合特徵。

In [ ]:
# 匯入必要資料科學與機器學習函式庫
import io
import gzip
import pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("✓ Libraries successfully imported.")

## [Phase 2] Data Understanding (資料理解與探索)
從指定 GitHub 網址爬取 50_Startups.csv 資料集，並進行敘述統計與散佈圖檢驗。

In [ ]:
# Step 1: 從網路獲取資料
url = 'https://raw.githubusercontent.com/Avik-Jain/100-Days-Of-ML-Code/master/datasets/50_Startups.csv'
response = requests.get(url)
data = pd.read_csv(io.StringIO(response.text))

# 檢視前 5 筆資料與欄位架構
print("Shape:", data.shape)
display(data.head())

# 統計摘要
display(data.describe())

# 檢查缺失值
print("Missing Values:\n", data.isnull().sum())

## [Phase 3] Data Preparation (資料前處理)
1. 切分特徵向量 $X$ 與目標標籤 $y$ (Profit)。
2. 名義類別變數 (State) 進行 One-Hot Encoding，並設定 `drop_first=True` 避免虛擬變數陷阱 (Dummy Variable Trap)。
3. 按照 80/20 比例切分訓練集與測試集。

In [ ]:
# 切分 X 與 y
X = data.drop('Profit', axis=1)
y = data['Profit']

# One-Hot Encoding 處理 State 欄位 (drop_first=True 避免共線性陷阱)
X_encoded = pd.get_dummies(X, columns=['State'], drop_first=True)
print("Encoded Features:", list(X_encoded.columns))

# 切分訓練集與測試集 (80/20, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=0)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## [Phase 4] Modeling (模型建置與特徵前向選取)
實作講義指定之 Forward Stepwise Selection 演算法，逐步評估 $k=1 \dots 5$ 個特徵時之表現。

In [ ]:
def evaluate_performance(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return rmse, r2

def forward_selection(X_tr, X_te, y_tr, y_te, max_features):
    selected_features = []
    results = []
    for i in range(1, max_features + 1):
        candidate_features = [f for f in X_tr.columns if f not in selected_features]
        candidate_rmse = np.inf
        candidate_r2 = -np.inf
        candidate_feature = None
        
        for feature in candidate_features:
            features_to_use = selected_features + [feature]
            model = LinearRegression()
            model.fit(X_tr[features_to_use], y_tr)
            y_pred = model.predict(X_te[features_to_use])
            current_rmse, current_r2 = evaluate_performance(y_te, y_pred)
            
            if current_rmse < candidate_rmse:
                candidate_rmse = current_rmse
                candidate_r2 = current_r2
                candidate_feature = feature
                
        selected_features.append(candidate_feature)
        results.append({
            'Number of Features': i,
            'Selected Features': selected_features.copy(),
            'RMSE': candidate_rmse,
            'R-squared': candidate_r2
        })
    return results

results = forward_selection(X_train, X_test, y_train, y_test, max_features=5)
results_df = pd.DataFrame(results)

## [Phase 5] Evaluation (模型評估與檢驗)
輸出如同講義要求之 Markdown 比較表格，展示 $k=1 \dots 5$ 特徵時之 RMSE 與 R-squared 變化。

In [ ]:
print(tabulate(results_df[['Number of Features', 'Selected Features', 'RMSE', 'R-squared']], headers='keys', tablefmt='pipe'))

# 繪製實際值 vs 預測值圖
best_features = ['R&D Spend', 'Marketing Spend']
best_model = LinearRegression()
best_model.fit(X_train[best_features], y_train)
y_pred_best = best_model.predict(X_test[best_features])

plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_best, color='#4f46e5', edgecolors='k', s=70, label='Test Startups')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')
plt.xlabel('Actual Profit ($)')
plt.ylabel('Predicted Profit ($)')
plt.title('Actual vs Predicted Profit (k=2 Optimal Model)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## [Phase 6] Deployment (模型序列化與部署)
使用 `pickle` 與 `gzip` 將最佳回歸模型保存為壓縮二進位檔案，供生產環境載入推論。

In [ ]:
# 序列化儲存
with gzip.open('startup_profit_model.pkl.gz', 'wb') as f:
    pickle.dump(best_model, f)
print("✓ Model successfully deployed and saved to startup_profit_model.pkl.gz")

# 驗證載入與即時推論
with gzip.open('startup_profit_model.pkl.gz', 'rb') as f:
    loaded_reg = pickle.load(f)

sample_input = pd.DataFrame([[165000.0, 470000.0]], columns=best_features)
pred_profit = loaded_reg.predict(sample_input)[0]
print(f"✓ Test Prediction for R&D=$165,000, Mkt=$470,000: ${pred_profit:,.2f}")